### Import relevant libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformer_model import TransformerModel

### Load dataset

In [2]:
df_action = pd.read_csv('../datasets/video_frame_features.csv')

In [3]:
df_action.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511,feature_512,video_file_name,label,subset
0,0.007248,-0.3200,0.3704,-0.006264,-0.1317,-0.1853,0.002575,-0.09660,-0.1758,-0.1566,...,-0.04224,-0.2285,0.1044,0.016660,0.2262,0.1467,0.2893,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
1,-0.009710,-0.3535,0.3982,-0.007336,-0.1371,-0.2051,0.008575,-0.08795,-0.2458,-0.1950,...,-0.10223,-0.2418,0.1360,0.000768,0.1517,0.1409,0.2942,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
2,0.015380,-0.3516,0.3926,0.001488,-0.1412,-0.1998,0.018110,-0.06890,-0.3164,-0.1958,...,-0.10270,-0.2330,0.1327,0.000418,0.2069,0.1462,0.2854,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
3,0.012540,-0.3513,0.3909,-0.000321,-0.1399,-0.2034,0.018300,-0.07210,-0.3160,-0.1935,...,-0.10205,-0.2317,0.1320,-0.001384,0.2039,0.1489,0.2844,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train
4,-0.007390,-0.3745,0.4067,-0.003576,-0.1442,-0.2162,0.007866,-0.05478,-0.3386,-0.2117,...,-0.07590,-0.1881,0.1242,-0.010920,0.1674,0.1584,0.2788,uccrime_Robbery126_x264_trimmed.mp4,abnormal,train


In [4]:
df_action.shape

(1323978, 515)

### Explore Data

In [6]:
# Check for missing values
df_action.isnull().sum()

feature_1          0
feature_2          0
feature_3          0
feature_4          0
feature_5          0
                  ..
feature_511        0
feature_512        0
video_file_name    0
label              0
subset             0
Length: 515, dtype: int64

In [7]:

# Check for label distribution
df_action['label'].value_counts()

abnormal    1051114
normal       272864
Name: label, dtype: int64

In [8]:
# Check number of unique video files per label
df_action.groupby('label')['video_file_name'].nunique()

label
abnormal    1079
normal      1069
Name: video_file_name, dtype: int64

In [9]:
# Check number of unieque video files
df_action['video_file_name'].nunique()

2148

In [10]:
# Check number of labels by subset
df_action.groupby('subset')['label'].value_counts()

subset  label   
test    abnormal    168217
        normal       40381
train   abnormal    674931
        normal      187512
val     abnormal    207966
        normal       44971
Name: label, dtype: int64

In [27]:
# Check number of unique video files per subset
df_action.groupby('subset')['video_file_name'].nunique()

subset
test      326
train    1496
val       326
Name: video_file_name, dtype: int64

In [11]:
df_action['subset'].unique()

array(['train', 'val', 'test'], dtype=object)

### Data preparation

In [12]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [13]:
# Helper function to convert series to supervised learning
def series_to_supervised(data, column_names, n_in=1, n_out=1, dropnan=True, n_out_index=None, is_drop_target=False):
    """
    convert series to supervised learning
    Params:
    - n_in: number of timesteps
    - n_out: number of output timestep for prediction | eg. 1 day, 2 days, ....
    - n_out_index: define specific variable for prediction
    - is_drop_target: drop target variable from input variables (to be used for classification problem)
    """
    df_new = pd.DataFrame(data)
    
    if is_drop_target:
        df_input = df_new.drop(columns=[column_names[n_out_index]])
    else:
        df_input = df_new.copy()
    
    n_vars = df_input.shape[1]
    
    cols, names = list(), list()
    
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df_input.shift(i))
        names += [f'{column_names[j]}(t-{i})' for j in range(n_vars)]
    
    # forecast sequence (t, t+1, ..., t+n)
    for i in range(0, n_out):
        if n_out_index:
            cols.append(df_new.shift(-i).iloc[:, n_out_index])
            if i == 0:
                names += [f'{column_names[n_out_index]}(t)']
            else:
                names += [f'{column_names[n_out_index]}(t+{i})']
        else:
            cols.append(df_new.shift(-i))
            if i == 0:
                names += [f'{column_names[j]}(t)' for j in range(n_vars)]
            else:
                names += [f'{column_names[j]}(t+{i})' for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values that generated from df_new.shift
    if dropnan:
        agg.dropna(inplace=True)
    
    return agg

In [14]:
# define features
column_names = df_action.columns.tolist()
column_names.remove('video_file_name')
column_names.remove('subset')
column_names

['feature_1',
 'feature_2',
 'feature_3',
 'feature_4',
 'feature_5',
 'feature_6',
 'feature_7',
 'feature_8',
 'feature_9',
 'feature_10',
 'feature_11',
 'feature_12',
 'feature_13',
 'feature_14',
 'feature_15',
 'feature_16',
 'feature_17',
 'feature_18',
 'feature_19',
 'feature_20',
 'feature_21',
 'feature_22',
 'feature_23',
 'feature_24',
 'feature_25',
 'feature_26',
 'feature_27',
 'feature_28',
 'feature_29',
 'feature_30',
 'feature_31',
 'feature_32',
 'feature_33',
 'feature_34',
 'feature_35',
 'feature_36',
 'feature_37',
 'feature_38',
 'feature_39',
 'feature_40',
 'feature_41',
 'feature_42',
 'feature_43',
 'feature_44',
 'feature_45',
 'feature_46',
 'feature_47',
 'feature_48',
 'feature_49',
 'feature_50',
 'feature_51',
 'feature_52',
 'feature_53',
 'feature_54',
 'feature_55',
 'feature_56',
 'feature_57',
 'feature_58',
 'feature_59',
 'feature_60',
 'feature_61',
 'feature_62',
 'feature_63',
 'feature_64',
 'feature_65',
 'feature_66',
 'feature_67',
 'fe

In [15]:
# Define timesteps (number of frames), number of features, and number of output step
timesteps = 30 # select only 10 fps from last 90 frames (3 seconds)
num_features = len(column_names[:-1])
n_steps_out = 1
n_out_index =  -1
n_obs = timesteps * num_features

In [24]:
# Reframe as supervised learning by each video file
subsets = df_action['subset'].unique()
df_by_subset = {}
skip_info = {}

for subset in subsets:
    df_action_subset = df_action[df_action['subset'] == subset]
    video_files = df_action_subset['video_file_name'].unique()

    df_tmp_list = []
    n_skip = 0
    for video_file in video_files:
        df_video = df_action_subset[df_action_subset['video_file_name'] == video_file]
        if df_video.shape[0] < timesteps: # Handle videos with frames less than 30
            # Skip these videos for now
            n_skip += 1
            continue
        elif df_video.shape[0] >= timesteps and df_video.shape[0] < 90: # Handle videos with frames between 30 and 90
            # Select evenly spaced 30 frames from available frames
            idx = np.linspace(0, df_video.shape[0] - 1, timesteps, dtype=int)
            df_selected = df_video.iloc[idx]

        elif df_video.shape[0] >= 90:
            # Select every 3rd frame 
            df_selected = df_video.iloc[::3]

        df_reframed = series_to_supervised(df_selected.drop(columns=['video_file_name', 'subset']), column_names, n_in=timesteps, n_out=n_steps_out, n_out_index=n_out_index, is_drop_target=True)
        df_tmp_list.append(df_reframed)
    
    skip_info[subset] = n_skip
    df_by_subset[subset] = pd.concat(df_tmp_list)

In [ ]:
# Check number of samples per subset
for key in df_by_subset.keys():
    print(f'{key}: {df_by_subset[key].shape}')

train: (244804, 15361)
val: (75004, 15361)
test: (60085, 15361)


In [ ]:
# Check of skipped videos per subset
skip_info

{'train': 13, 'val': 4, 'test': 0}

In [ ]:
# Rename target column from 'label(t)' to 'label'
for key in df_by_subset.keys():
    df_by_subset[key].rename(columns={'label(t)': 'label'}, inplace=True)

In [ ]:
# Check number of samples per label for each subset
for key in df_by_subset.keys():
    print(f'- {key}:\n{df_by_subset[key]["label"].value_counts()}')

- train:
abnormal    203321
normal       41483
Name: label, dtype: int64
- val:
abnormal    64511
normal      10493
Name: label, dtype: int64
- test:
abnormal    51252
normal       8833
Name: label, dtype: int64


In [35]:
# Since video has different length, number of samples per label is imbalanced.
# Here, we need to balance number of samples for each label in each subset
df_train = pd.concat([df_by_subset['train'][df_by_subset['train']['label'] == label].sample(n=df_by_subset['train']['label'].value_counts().min(), random_state=SEED) for label in df_by_subset['train']['label'].unique()])

In [36]:
df_train['label'].value_counts()

abnormal    41483
normal      41483
Name: label, dtype: int64

In [37]:
df_val = pd.concat([df_by_subset['val'][df_by_subset['val']['label'] == label].sample(n=df_by_subset['val']['label'].value_counts().min(), random_state=SEED) for label in df_by_subset['val']['label'].unique()])
df_val['label'].value_counts()

abnormal    10493
normal      10493
Name: label, dtype: int64

In [38]:
df_test = pd.concat([df_by_subset['test'][df_by_subset['test']['label'] == label].sample(n=df_by_subset['test']['label'].value_counts().min(), random_state=SEED) for label in df_by_subset['test']['label'].unique()])
df_test['label'].value_counts()

abnormal    8833
normal      8833
Name: label, dtype: int64

In [40]:
df_train['label'] = df_train['label'].map({'abnormal': 0, 'normal': 1})
df_val['label'] = df_val['label'].map({'abnormal': 0, 'normal': 1})
df_test['label'] = df_test['label'].map({'abnormal': 0, 'normal': 1})

In [ ]:
# Reshape data into 3D array [samples, timesteps, features]

X_train = df_train.drop(columns=['label']).values.reshape((-1, timesteps, num_features))
y_train = df_train['label'].values.reshape(-1, 1)

X_val = df_val.drop(columns=['label']).values.reshape((-1, timesteps, num_features))
y_val = df_val['label'].values.reshape(-1, 1)

X_test = df_test.drop(columns=['label']).values.reshape((-1, timesteps, num_features))
y_test = df_test['label'].values.reshape(-1, 1)

In [51]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((82966, 30, 512),
 (82966, 1),
 (20986, 30, 512),
 (20986, 1),
 (17666, 30, 512),
 (17666, 1))

In [52]:
# Create data loaders
batch_size = 32
train_loader = DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(list(zip(X_val, y_val)), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)

In [44]:
# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

### Train model using Transformer architecture based on PyTorch

In [45]:
import copy

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_val_acc = -np.inf
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None
        self.best_optimizer_state = None
        self.best_epoch = None

    def __call__(self, val_acc, model, optimizer, epoch):
        if val_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = val_acc
            self.best_model_state = copy.deepcopy(model.state_dict())
            self.best_optimizer_state = copy.deepcopy(optimizer.state_dict())
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True

    def restore_best_model(self, model, optimizer):
        print('Restoring best model...')
        model.load_state_dict(self.best_model_state)
        optimizer.load_state_dict(self.best_optimizer_state) 

In [46]:
# define hyperparameters
input_dim = num_features
d_model = 256
num_heads = 8
num_layers = 2
dim_feedforward = 128
output_dim = 1 # binary classification with sigmoid activation
tl_dropout = 0.1 # transformer layer dropout
nn_dropout = 0.3 # neural network dropout
lr = 0.0001

In [47]:
# Initialize model
model = TransformerModel(input_dim, d_model, num_heads, num_layers, dim_feedforward, output_dim, tl_dropout, nn_dropout).to(device)
model

TransformerModel(
  (embedding): Linear(in_features=512, out_features=256, bias=True)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (linear1): Linear(in_features=256, out_features=128, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=128, out_features=256, bias=True)
    (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (

In [48]:
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
early_stopping = EarlyStopping(patience=20, min_delta=0.0001)

# Schedule learning rate
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, verbose=True)

In [ ]:
# train model
num_epochs = 100
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for X_train_batch, y_train_batch in train_loader:
        X_train_batch, y_train_batch = X_train_batch.to(device).float(), y_train_batch.to(device).float()
        
        # forward pass
        outputs = model(X_train_batch)
        loss = criterion(outputs, y_train_batch)
        
        # backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for X_valid_batch, y_valid_batch in val_loader:
            X_valid_batch, y_valid_batch = X_valid_batch.to(device).float(), y_valid_batch.to(device).float()
            outputs = model(X_valid_batch)
            v_loss = criterion(outputs, y_valid_batch)
            val_loss += v_loss.item()
            probs = torch.sigmoid(outputs)
            val_acc += (probs.round() == y_valid_batch).float().mean()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    val_acc /= len(val_loader)
    
    scheduler.step(val_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train_loss: {train_loss:.6f}, Val_loss: {val_loss:.6f}, Val_acc: {val_acc:.6f}')
    early_stopping(val_acc, model, optimizer, epoch+1)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break

# load best model
early_stopping.restore_best_model(model, optimizer)
print(f'Best epoch: {early_stopping.best_epoch} with val_acc: {early_stopping.best_val_acc:.6f}')